In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, pandas as pd

DATA_DIR = Path("/home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/")

FILES = {
    "physio_data_unstandardized": "nature_filtered_nonan_cfc.csv",
    "genoprobs_proj_standardized": "genoprobs_rescaled.csv",
    "cfc_data": "cfc_data_filtered.csv",
    "standardized_combined": "do_mice_data_standardized.csv",
}

MANIFEST_CSV = DATA_DIR / "hash_manifest.csv"
HASHES_PY    = DATA_DIR / "expected_hashes.py"

# --- Helper: SHA-256 ---
def sha256_file(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

# --- Compute hashes & assemble DataFrame ---
rows = []
for stage, fname in FILES.items():
    path = DATA_DIR / fname
    if not path.exists():
        rows.append({
            "stage": stage, "filename": fname, "relative_path": fname,
            "size_bytes": None, "modified_utc": None, "sha256": None,
            "status": "MISSING",
        })
    else:
        st = path.stat()
        rows.append({
            "stage": stage,
            "filename": fname,
            "relative_path": fname,
            "size_bytes": st.st_size,
            "modified_utc": datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat(),
            "sha256": sha256_file(path),
            "status": "OK",
        })

df_hashes = pd.DataFrame(rows, columns=[
    "stage","filename","relative_path","size_bytes","modified_utc","sha256","status"
])

# --- Write manifest CSV ---
df_hashes.to_csv(MANIFEST_CSV, index=False)

# --- Write importable Python file with DIR, FILES, EXPECTED_SHA256 ---
expected_dict = {r.stage: r.sha256 for r in df_hashes.itertuples(index=False) if r.status=="OK"}
py_content = f"""# Auto-generated by hash_data.ipynb
from pathlib import Path

# Folder containing the four CSVs
DATA_DIR = Path(r"{str(DATA_DIR)}")

# Stage -> filename mapping
FILES = {{
    "physio_data_unstandardized": "nature_filtered_nonan_cfc.csv",
    "genoprobs_proj_standardized": "genoprobs_rescaled.csv",
    "cfc_data": "cfc_data_filtered.csv",
    "standardized_combined": "do_mice_data_standardized.csv",
}}

# Expected SHA-256 digests (byte-level)
EXPECTED_SHA256 = {expected_dict!r}
"""

HASHES_PY.write_text(py_content)

# --- Display results ---
print(f"Wrote manifest  -> {MANIFEST_CSV}")
print(f"Wrote hashes py -> {HASHES_PY}")
df_hashes

Wrote manifest  -> /home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/hash_manifest.csv
Wrote hashes py -> /home/rachel/Desktop/mm-vae data/caloric_restriction_DO_mice/expected_hashes.py


,stage,filename,relative_path,size_bytes,modified_utc,sha256,status
0,physio_data_unstandardized,nature_filtered_nonan_cfc.csv,nature_filtered_nonan_cfc.csv,315698,2026-02-19T04:12:14.533305+00:00,2cd1b54b987daf55f57f443b767f014bca7fc4fbb6b3e3...,OK
1,genoprobs_proj_standardized,genoprobs_rescaled.csv,genoprobs_rescaled.csv,109103866,2026-03-06T22:47:11.530596+00:00,bc370b31fa103d3d3d235c17b5bc5f8cb166d8c719f3da...,OK
2,cfc_data,cfc_data_filtered.csv,cfc_data_filtered.csv,10018,2026-02-19T04:12:32.588130+00:00,cb9ee095c72779f5313ebb033f95225c9fbfd4498aa286...,OK
3,standardized_combined,do_mice_data_standardized.csv,do_mice_data_standardized.csv,57152380,2026-08-20T14:15:12.671608+00:00,e524b27951692ee4155cc89ae2428de51c384eb75c6ea7...,OK
